### RDR 2 PDF RAG ###

In [1]:
import os
from dotenv import load_dotenv
load_dotenv(override = True)

True

In [2]:
if os.environ["OPENAI_API_KEY"] is None:
    raise ValueError("OPENAI_API_KEY environment variable is not set.")
else:
    print("OPENAI_API_KEY is set.")

OPENAI_API_KEY is set.


In [3]:
from langchain_openai import ChatOpenAI

In [4]:
llm = ChatOpenAI(model = "gpt-5-nano", temperature = 0)

In [5]:
response = llm.invoke('Hello, how are you?')
response.content

'Hi there! I’m here and ready to help with whatever you need. How can I assist you today? If you’d like, we can chat, solve a problem, plan something, or I can share a fun fact.'

## RAG WITH PDF ##


### STEP 1: Etracting from pdf ###

In [6]:
from langchain_community.document_loaders import PyMuPDFLoader

pdf_path = './Red Dead Redemption 2 - The Complete Official Guide.pdf'

loader = PyMuPDFLoader(pdf_path)

docs = loader.load()

docs


C:\Users\Smit\AppData\Local\Temp\ipykernel_6324\1027836647.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


[Document(metadata={'producer': 'Adobe Acrobat Pro DC 19 Paper Capture Plug-in', 'creator': 'Adobe Acrobat 11.0', 'creationdate': '2018-11-04T00:00:45-07:00', 'source': './Red Dead Redemption 2 - The Complete Official Guide.pdf', 'file_path': './Red Dead Redemption 2 - The Complete Official Guide.pdf', 'total_pages': 355, 'format': 'PDF 1.6', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2018-11-04T21:34:40-08:00', 'trapped': '', 'modDate': "D:20181104213440-08'00'", 'creationDate': "D:20181104000045-07'00'", 'page': 0}, page_content=''),
 Document(metadata={'producer': 'Adobe Acrobat Pro DC 19 Paper Capture Plug-in', 'creator': 'Adobe Acrobat 11.0', 'creationdate': '2018-11-04T00:00:45-07:00', 'source': './Red Dead Redemption 2 - The Complete Official Guide.pdf', 'file_path': './Red Dead Redemption 2 - The Complete Official Guide.pdf', 'total_pages': 355, 'format': 'PDF 1.6', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2018-11-04T21:34

In [7]:
for i in docs:
    i.metadata = {'source' : 'rdr2_complete_guide',
                  'developer' : 'Rockstar Games',
                  } 

### STEP 2 : Chunks

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100)

chunks = splitter.split_documents(docs)

print('The size of the chunks is:', len(chunks))


The size of the chunks is: 1058


In [9]:
chunks[0].metadata

{'source': 'rdr2_complete_guide', 'developer': 'Rockstar Games'}

### STEP 3 : Embedding model

In [10]:
from langchain_openai import OpenAIEmbeddings

embedding = OpenAIEmbeddings(model = "text-embedding-3-small")

### STEP 4 : Storing 

In [11]:
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks, 
    embedding=embedding,
    persist_directory="./VectorDB/"
)

len(vectorstore)

3492

In [12]:
##  STEP 5 : Semantic Search

In [13]:
context = vectorstore.similarity_search("What is the best weapon in RDR2?", k=2)

display(context)

[Document(metadata={'source': 'rdr2_complete_guide', 'developere': 'Rockstar Games'}, page_content="rifles, but lower accuracy and damage. \n* Sniper rifles grant excellent damage and precision over long \ndistances. \n* Shotguns shoot a batch of buckshot. They are effective only when \na fight is practically toe-to-toe. though they are peerless when it \ncomes to clearing a room or holding a narrow choke point. They are \neven more impactful with slug ammunition. \n* The bow is a silent and accurate weapon, ideal for stealth kills and \nhunting. Various kinds of arrows can be crafted and used for specific \nbenefits. \n* Melee weapons are perfect for brawls and situations where it's not \npossible to carry a gun. You can fight with your fists, various knives, \nand even hatchets and swords. \n* Throwables such as throwing knives and tomahawks are an \nalternative to the bow if you need to eliminate a target quietly. \nIf stealth is not required, volatile area-of-effect weapons such as

## Talk to LLM

In [14]:
response = llm.invoke(f'What is the best weapon in RDR2? Please use this context:{context}')
response.content

'The guide you provided doesn’t name one single “best” weapon. It describes several weapons as best for different roles, so the best choice depends on how you want to play:\n\n- Sniper rifles: Excellent damage and precision at long range. If you want the strongest long-range option, this is the standout.\n- Shotguns: Very powerful up close, great for clearing rooms or holding tight chokepoints, especially with slug ammo.\n- Bow: Silent and accurate, ideal for stealth kills and hunting; arrows can be crafted for various benefits.\n- Melee weapons: Good for brawls or situations where you can’t carry a gun.\n- Throwables (knives, tomahawks): Quiet eliminations as an alternative to the bow.\n- Fire bottles and dynamite: Strong for crowd control and close-range AoE.\n\nBottom line: the “best” weapon is situational. If you want a strong all-around option with long-range power, a sniper rifle is a top pick from this guide; otherwise, pick based on whether you’re stealthy (bow), close-quarters